In [1]:
%config Completer.use_jedi = False

In [2]:
# load module
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorboard as tb 
import pytorch_lightning as pl


from pathlib import Path
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from IPython.core.display import HTML

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']

from my_funs import *

/home/nplab/.local/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /home/nplab/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will n

In [3]:
lrs = [0.1 ,0.01 , 0.001]
optims = [ 'Adam' , 'AdamW', 'Adamax' , 'NAdam', 'RAdam' ,'ranger']

funs = [moving_average_alpha, moving_average_com , moving_average_span
        , moving_average_halflife]
ewma_factor = np.arange(1,10)
from itertools import product
result_df = pd.DataFrame(product(ewma_factor, lrs , optims, funs))
result_df.columns = ['ewma_factor' , 'lr' , 'optim' ,'ewma' ]
result_df['str_ewma'] = result_df['ewma'].astype(str) 
alpha_index = result_df[result_df['str_ewma'].str.contains('alpha')].index
result_df.loc[alpha_index , 'ewma_factor'] /= 10

result_df

,ewma_factor,lr,optim,ewma,str_ewma
0,0.1,0.100,Adam,<function moving_average_alpha at 0x7f49a43c1e50>,<function moving_average_alpha at 0x7f49a43c1e50>
1,1.0,0.100,Adam,<function moving_average_com at 0x7f49a439a4c0>,<function moving_average_com at 0x7f49a439a4c0>
2,1.0,0.100,Adam,<function moving_average_span at 0x7f49a43b2280>,<function moving_average_span at 0x7f49a43b2280>
3,1.0,0.100,Adam,<function moving_average_halflife at 0x7f49a43...,<function moving_average_halflife at 0x7f49a43...
4,0.1,0.100,AdamW,<function moving_average_alpha at 0x7f49a43c1e50>,<function moving_average_alpha at 0x7f49a43c1e50>
...,...,...,...,...,...
643,9.0,0.001,RAdam,<function moving_average_halflife at 0x7f49a43...,<function moving_average_halflife at 0x7f49a43...
644,0.9,0.001,ranger,<function moving_average_alpha at 0x7f49a43c1e50>,<function moving_average_alpha at 0x7f49a43c1e50>
645,9.0,0.001,ranger,<function moving_average_com at 0x7f49a439a4c0>,<function moving_average_com at 0x7f49a439a4c0>
646,9.0,0.001,ranger,<function moving_average_span at 0x7f49a43b2280>,<function moving_average_span at 0x7f49a43b2280>


In [4]:
model_ckpt_df = pd.DataFrame(columns = ['model_ckpt'])

In [ ]:
# ewma, lr, optim은 이전 TEST에서 구한 최적의 값 사용
pl.seed_everything(42)
batch_size=32

for idx in result_df.index[528:]:
    factor = result_df['ewma_factor'].loc[idx]
    funs = result_df['ewma'].loc[idx]
    lr = result_df['lr'].loc[idx]
    optim = result_df['optim'].loc[idx]
    
    
    
    data = data_processing('../../train.csv' , factor ,funs )
    training = get_training(data,24, 24*7)

    dataloader = training.to_dataloader(batch_size)
    validation = TimeSeriesDataSet.from_dataset(training, data, predict=True, stop_randomization=True)
    train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size*100, num_workers=0)

    early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
    lr_logger = LearningRateMonitor()  
    logger = TensorBoardLogger('logger')
    trainer = pl.Trainer(
        max_epochs=100,
        gpus=1,
        gradient_clip_val=0.1,
        limit_train_batches=20,  
        callbacks=[lr_logger],
        logger=logger,
    )


    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=lr,
        hidden_size=32 ,#hidden,  # most important hyperparameter apart from learning rate
        attention_head_size=4,
        dropout=0.1,  # between 0.1 and 0.3 are good values
        hidden_continuous_size=32,#hidden,  # set to <= hidden_size
        output_size=7,  # 7 quantiles by default
        loss=QuantileLoss(),
        # reduce learning rate if no improvement in validation loss after x epochs
        reduce_on_plateau_patience=4,
    )

    tft.hparams['optimizer'] = optim

    trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
    )

    print(trainer.checkpoint_callback.best_model_path)
    model_ckpt_df.loc[idx, 'model_ckpt'] = trainer.checkpoint_callback.best_model_path
    model_ckpt_df.to_csv('result_model_ckpt.csv')

Global seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 538   
3  | prescalers                         | ModuleDict                      | 448   
4  | static_variable_selection          | VariableSelectionNetwork        | 4.6 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 29.8 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 10.0 K
7  | static_context_variable_s

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [ ]:
model_ckpt_df